## Pandas vs PySpark Comparison

Reads only the CSV files in `rq1_pandas/` and `rq1_spark/`. Counts must match exactly; rates within 0.01; percentiles and medians within 1% (Spark percentiles are approximate); means within 0.1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/hmda/"
P, S = BASE + "rq1_pandas/", BASE + "rq1_spark/"

# (file, check name, key column, columns to compare, absolute tolerance, relative tolerance)
CHECKS = [
    ("R1_outcomes.csv",            "Application outcomes",              "outcome",       ["count"],                           0,     0),
    ("R2_denial_by_loan_type.csv", "Denial rate by loan type", "type_label",    ["applications", "denial_rate"],     0.01,  0),
    ("R2_denial_by_purpose.csv",   "Denial rate by purpose",   "purpose_label", ["applications", "denial_rate"],     0.01,  0),
    ("R3_distribution_stats.csv",  "Count (n)",               "variable",      ["n"],                               0,     0),
    ("R3_distribution_stats.csv",  "Percentiles (±1%)",              "variable",      ["p1", "Q1", "median", "Q3", "p99"], 0,     0.01),
    ("R3_distribution_stats.csv",  "Mean",                       "variable",      ["mean"],                            0.1,   0),
    ("R3_median_by_loan_type.csv", "Median by loan type (±1%)",     "type_label",    ["loan_amount", "income"],           0,     0.01),
    ("R4_mix_purpose_x_type.csv",  "Mix purpose × type",      None,            None,                                0.011, 0),
]

rows, details = [], []
for f, name, key, cols, atol, rtol in CHECKS:
    if not (os.path.exists(P + f) and os.path.exists(S + f)):
        rows.append({"check": name, "file": f, "rows matched": "—", "all match": "MISSING FILE"})
        continue
    p, s = pd.read_csv(P + f), pd.read_csv(S + f)
    if key is None:                                   # R4 cross-tab: the first column is the key
        key = p.columns[0]
        s = s.rename(columns={s.columns[0]: key})
        cols = [c for c in p.columns if c != key]
    m = p[[key] + cols].merge(s[[key] + cols], on=key, how="outer", suffixes=("_pandas", "_spark"))
    ok = pd.Series(True, index=m.index)
    for c in cols:
        ok &= np.isclose(m[f"{c}_pandas"].astype(float), m[f"{c}_spark"].astype(float),
                         atol=atol, rtol=rtol)
    m["match"] = ok
    details.append((name, m))
    rows.append({"check": name, "file": f,
                 "rows matched": f"{int(ok.sum())}/{len(m)}", "all match": bool(ok.all())})

summary = pd.DataFrame(rows)
summary.to_csv(P + "RQ1_compare_pandas_spark.csv", index=False)
display(summary)

done = summary[summary["all match"].apply(lambda x: isinstance(x, bool))]
print(f"\nChecked {len(done)}/{len(summary)} checks | "
      f"RQ1 all match: {done['all match'].all() if len(done) else '—'}")

# Show rows that do not match (if any)
for name, m in details:
    if not m["match"].all():
        print(f"\n--- Mismatch: {name} ---")
        display(m[~m["match"]])

Mounted at /content/drive


,check,file,rows matched,all match
0,Application outcomes,R1_outcomes.csv,8/8,True
1,Denial rate by loan type,R2_denial_by_loan_type.csv,4/4,True
2,Denial rate by purpose,R2_denial_by_purpose.csv,6/6,True
3,Count (n),R3_distribution_stats.csv,2/2,True
4,Percentiles (±1%),R3_distribution_stats.csv,2/2,True
5,Mean,R3_distribution_stats.csv,2/2,True
6,Median by loan type (±1%),R3_median_by_loan_type.csv,4/4,True
7,Mix purpose × type,R4_mix_purpose_x_type.csv,7/7,True



Checked 8/8 checks | RQ1 all match: True
